#### Objectif :

Ce notebook met en place une seconde étape d'analyse, complémentaire au pipeline Morlet exploratoire.

L'idée générale est la suivante :

on utilise Morlet pour repérer de manière exploratoire les bandes et fenêtres fréquentielles potentiellement modulées ;
on revient ensuite au signal continu, que l'on filtre dans des sous-bandes fixes ;
on extrait l'enveloppe d'amplitude par transformée de Hilbert ;
on normalise cette enveloppe à l'échelle de la session ;
on époque autour des stimulations ;
on regroupe les observations par condition cognitive et localité (local / distant) ;
on teste ensuite les courbes temporelles Hilbert, session par session ou pooled across sessions.

Le découpage suit la logique méthodologique inspirée de Vidal et al. (2010), adaptée aux stimulations intracérébrales.

In [ ]:
# imports
from pathlib import Path
import pandas as pd
import json

from utils_time_frequency.lfp_hilbert_utils import (
    HilbertConfig,
    # build_hilbert_subbands,
    build_main_band_to_subbands,
    prepare_hilbert_session_data,
    run_session_hilbert,
    run_all_sessions_hilbert,
    load_hilbert_session_exports,
    load_hilbert_band_epochs,
)

from utils_time_frequency.lfp_hilbert_stats import (
    HilbertStatsConfig,
    stack_hilbert_band_condition_locality,
    stack_hilbert_band_condition_locality_across_sessions,
    run_hilbert_session_condition_stats,
    run_pooled_hilbert_condition_stats,
    run_all_hilbert_stats,
)

from utils_time_frequency.lfp_preprocess_utils import (
    list_trc_sessions,
    load_bad_channels_table,
)

from utils_time_frequency.lfp_morlet_stats import (
    build_main_condition_index,
    build_cog_subcategory_index,
)

#### Hilbert run

Structure de travail : 

root_dir/                                                                {= root_dir, dans HilbertConfig}
    └── results_hilbert/                                                 {= output_dir, dans HilbertConfig}
        └── PATIENT_stimN/                                               {créé}
            ├── PATIENT_stimN_metadata.json
            ├── PATIENT_stimN_trial_table.csv
            ├── PATIENT_stimN_times.npy
            └── PATIENT_stimN_hilbert_<band>.npy {theta, alpha, beta, low_gamma}
        ...
        └── run_summary_hilbert.json                                     {créé, quand run Hilbert}


    ├── PATIENT_stimN.TRC                                                 {nécessaire}
    ├── PATIENT_stimN_stim_events_TRC_corrected.txt                       {nécessaire, sinon créé}
    ├── PATIENT_stimN_stim_events_TRC_re-shifted_loca_COG.txt             {nécessaire}

    ou si pas de *_corrected.txt (normalement si, car on a deja run morlet sur cette session ?), alors besoin de :
    ├── PATIENT_stimN_stim_events_TRC.txt                                 {optionnel}
    ├── PATIENT_stimN_stim_sevents_TRC_shifted.txt                        {optionnel}
    ├── PATIENT_stimN_stim_events_TRC_re-shifted.txt                      {optionnel}
    ...
    └── TRC_bad_channels.xlsx                                             {nécessaire}

In [6]:
# configuration Hilbert

hilbert_cfg = HilbertConfig(
    root_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog",
    output_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert",

    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,

    do_notch=True,
    notch_freqs=(50.0, 100.0, 150.0),
    notch_q=30.0,
    do_highpass=True,
    highpass_hz=0.1,

    gamma_low_hz=30.0,
    gamma_high_hz=150.0,
    gamma_step_hz=10.0,

    beta_low_hz = 12.0,
    beta_high_hz = 30.0,
    beta_step_hz = 4.0,

    alpha_low_hz = 8.0,
    alpha_high_hz = 12.0,
    alpha_step_hz = 2.0,

    theta_low_hz=4.0,
    theta_high_hz=8.0,
    theta_step_hz=2.0,

    include_delta=False,
    delta_low_hz=1.0,
    delta_high_hz=4.0,
    delta_step_hz=1.0,

    normalization_mode="pre_first_stim_mean",
    target_dt_ms=16.0,

    save_subband_epochs=False,
    save_main_band_epochs=True,
    verbose=True,
)

In [7]:
# inspection des sous-bandes Hilbert construites
main_band_map = build_main_band_to_subbands(hilbert_cfg)
main_band_map

{'theta': [('theta', 4.0, 6.0), ('theta', 6.0, 8.0)],
 'alpha': [('alpha', 8.0, 10.0), ('alpha', 10.0, 12.0)],
 'beta': [('beta', 12.0, 16.0),
  ('beta', 16.0, 20.0),
  ('beta', 20.0, 24.0),
  ('beta', 24.0, 28.0),
  ('beta', 28.0, 30.0)],
 'low_gamma': [('gamma', 30.0, 40.0),
  ('gamma', 40.0, 50.0),
  ('gamma', 50.0, 60.0),
  ('gamma', 60.0, 70.0),
  ('gamma', 70.0, 80.0)],
 'high_gamma': [('gamma', 80.0, 90.0),
  ('gamma', 90.0, 100.0),
  ('gamma', 100.0, 110.0),
  ('gamma', 110.0, 120.0),
  ('gamma', 120.0, 130.0),
  ('gamma', 130.0, 140.0),
  ('gamma', 140.0, 150.0)]}

In [8]:
# sessions disponibles
root_dir = Path(hilbert_cfg.root_dir)
sessions = list_trc_sessions(root_dir)
print(f"{len(sessions)} sessions trouvées")
sessions

1 sessions trouvées


['P119_FM71_stim4']

In [9]:
# chargement des bad channels
bad_df = load_bad_channels_table(root_dir)
bad_df

,session,bad_channels
0,P64_BR34_stim2,"B14,C15,GC6,GC7,W46"
1,P97_BM50_stim1,GC'4
2,P101_DC54_stim2,"B_1,H_11,CU_12,Bp8"
3,P101_DC54_stim4,B_1
4,P119_FM71_stim4,"Bp1,Bp2,Bp3,Bp6,Bp7,Bp8"


##### Hilbert / une session

In [ ]:
session = sessions[0]

session_data = prepare_hilbert_session_data(
    session=session,
    root_dir=root_dir,
    bad_df=bad_df,
    cfg=hilbert_cfg,
) # crée le data_bp, le df avec infos stims, etc

In [16]:
# structure préparée
print("Session :", session_data["session"])
print("sfreq :", session_data["sfreq"])
# print("n_trials :", len(session_data["trials_df"]))
print("n_bipolar_channels :", len(session_data["bp_names"]))

session_data#["trials_df"][[
#     "label_stim",
#     "group_label",
#     "cog_labels",
#     "stim_shaft",
# ]]

Session : P119_FM71_stim4
sfreq : 512.0
n_bipolar_channels : 85


{'session': 'P119_FM71_stim4',
 'sfreq': 512.0,
 'stims_df':                       label_stim   t_start  duration         lobe  \
 0      Tp3-Tp42.0mA7.0Hz1025µsec   307.656   9.86499     L Insula   
 1      Tp5-Tp62.0mA7.0Hz1025µsec   357.097   9.87872   L Temporal   
 2      Tp8-Tp92.0mA7.0Hz1025µsec   391.859   9.88953   L Temporal   
 3     Hp9-Hp102.0mA7.0Hz1025µsec   445.130   9.86789   L Temporal   
 4      Hp5-Hp62.0mA7.0Hz1025µsec   474.216   9.86789   L Temporal   
 5      Hp4-Hp52.0mA7.0Hz1025µsec   528.119   9.87872   L Temporal   
 6      Hp1-Hp22.0mA7.0Hz1025µsec   604.054   9.88953   L Temporal   
 7      Bp6-Bp72.0mA7.0Hz1025µsec   670.108   9.88953   L Temporal   
 8     CP1-CP21.2mA50.0Hz1025µsec   927.946   4.99887  R Cingulate   
 9       X7-X81.2mA50.0Hz1025µsec   996.349   4.98804     R Insula   
 10      X5-X61.2mA50.0Hz1025µsec  1021.979   4.95560     R Insula   
 11      X3-X41.2mA50.0Hz1025µsec  1046.349   4.93393     R Insula   
 12      X3-X41.2mA50.0Hz1025µ

In [ ]:
# run Hilbert sur cette session
session_out = run_session_hilbert( # 56 min sur une session à 15 stims (mais ensuite, 30 min pour une session à 30 stims...)
    session_data=session_data,
    out_dir=Path(hilbert_cfg.output_dir),
    cfg=hilbert_cfg,
)

session_out


=== Hilbert session P119_FM71_stim4 ===
[INFO] P119_FM71_stim4: facteur de décimation Hilbert = 8
[OK] P119_FM71_stim4: résultats Hilbert sauvegardés dans /home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert/P119_FM71_stim4


PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert/P119_FM71_stim4')

In [19]:
# reload les exports Hilbert de cette session
exports = load_hilbert_session_exports(session_out)

print("Session exportée :", exports["session"])
print("times shape :", exports["times"].shape)
exports

Session exportée : P119_FM71_stim4
times shape : (384,)


{'session': 'P119_FM71_stim4',
 'metadata': {'session': 'P119_FM71_stim4',
  'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog',
   'output_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert',
   'pre_length': 3.0,
   'post_length': 3.0,
   'epsilon': 0.1,
   'do_notch': True,
   'notch_freqs': [50.0, 100.0, 150.0],
   'notch_q': 30.0,
   'do_highpass': True,
   'highpass_hz': 0.1,
   'gamma_low_hz': 30.0,
   'gamma_high_hz': 150.0,
   'gamma_step_hz': 10.0,
   'beta_low_hz': 12.0,
   'beta_high_hz': 30.0,
   'beta_step_hz': 4.0,
   'alpha_low_hz': 8.0,
   'alpha_high_hz': 12.0,
   'alpha_step_hz': 2.0,
   'theta_low_hz': 4.0,
   'theta_high_hz': 8.0,
   'theta_step_hz': 2.0,
   'include_delta': False,
   'delta_low_hz': 1.0,
   'delta_high_hz': 4.0,
   'delta_step_hz': 1.0,
   'main_bands': ['delta',
    'theta',
    'alpha',
    'beta',
    'low_gamma',
    'high_gamma'],
   'low_gamma_s

In [20]:
# reload une bande Hilbert de la session
band_name = "low_gamma"
arr = load_hilbert_band_epochs(session_out, exports["session"], band_name)
print(arr.shape)   # attendu : (n_trials, n_channels, n_times)

(31, 85, 384)


##### Hilbert / toutes les sessions

In [6]:
summary_hilbert = run_all_sessions_hilbert(hilbert_cfg) # 243 min pour 2 sessions avec 60n de stims
summary_hilbert

1 sessions TRC trouvées pour Hilbert

=== Préparation Hilbert session P101_DC54_stim2 ===


[INFO] P101_DC54_stim2.TRC: sfreq = 2048.0
[INFO] P101_DC54_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']

=== Hilbert session P101_DC54_stim2 ===
[INFO] P101_DC54_stim2: facteur de décimation Hilbert = 33
[OK] P101_DC54_stim2: résultats Hilbert sauvegardés dans /home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_hilbert/P101_DC54_stim2


{'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog',
  'output_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_hilbert',
  'pre_length': 3.0,
  'post_length': 3.0,
  'epsilon': 0.1,
  'do_notch': True,
  'notch_freqs': (50.0, 100.0, 150.0),
  'notch_q': 30.0,
  'do_highpass': True,
  'highpass_hz': 0.1,
  'gamma_low_hz': 30.0,
  'gamma_high_hz': 150.0,
  'gamma_step_hz': 10.0,
  'beta_low_hz': 12.0,
  'beta_high_hz': 30.0,
  'beta_step_hz': 4.0,
  'alpha_low_hz': 8.0,
  'alpha_high_hz': 12.0,
  'alpha_step_hz': 2.0,
  'theta_low_hz': 4.0,
  'theta_high_hz': 8.0,
  'theta_step_hz': 2.0,
  'include_delta': False,
  'delta_low_hz': 1.0,
  'delta_high_hz': 4.0,
  'delta_step_hz': 1.0,
  'main_bands': ('delta', 'theta', 'alpha', 'beta', 'low_gamma', 'high_gamma'),
  'low_gamma_split': (30.0, 80.0),
  'high_gamma_split': (80.0, 150.0),
  'alpha_range': (8.0, 12.0),
  'beta_range': (12.0, 30.0),
  'theta_rang

In [ ]:
# lecture des résumés produits
summary_file = Path(hilbert_cfg.output_root) / "run_summary_hilbert.json"
with open(summary_file, "r") as f:
    summary = json.load(f)
summary

#### Stats sur Hilbert

Structure de travail : 

results_hilbert/                                                     {= input_root, dans HilbertStatsConfig}
    └── PATIENT_stimN/                                               {nécessaire}
        ├── PATIENT_stimN_metadata.json
        ├── PATIENT_stimN_trial_table.csv
        ├── PATIENT_stimN_times.npy
        └── PATIENT_stimN_hilbert_<band>.npy {theta, alpha, beta, low_gamma}
    ...
    └── run_summary_hilbert.json                                     {nécessaire}


├── results_hilbert_stats/                                           {= output_root, dans HilbertStatsConfig}
    └── pooled_across_sessions/                                      {créé}
        ├── condition_main/
            ├── negatif/
                ├── distant/
                    ...
                    └── <band>
                        ├── cluster_pvals.npy
                        ├── mean_trace.npy
                        ├── median_trace.npy
                        ├── pvals_wilcoxon.npy
                        ├── pvals_wilcoxon_fdr.npy
                        ├── sig_mask_cluster.npy
                        ├── sig_mask_wilcoxon_fdr.npy
                        ├── stat_wilcoxon.npy
                        ├── T_obs_cluster.npy
                        ├── observations_table.csv
                        ├── figure_cluster.png
                        └── figure_wilcoxon.png
                └── local
                    ...idem que distant
            ...
        ├── condition_subcategories/
            ...idem que condition_main
        ├── hilbert_stats_config.json
        └── summary_hilbert_stats.csv
    ...
    └── run_summary_hilbert.json                                     {créé}


In [2]:
# configuration stats

hilbert_stats_cfg = HilbertStatsConfig(
    input_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert",
    output_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert_stats",

    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),

    run_wilcoxon_fdr=True,
    run_cluster_perm=True,
    alpha_fdr=0.05,

    cluster_alpha=0.05,
    n_permutations=2000,
    cluster_threshold_p=0.05,
    tail=0,
    seed=13,

    min_trials_per_condition=3,
    make_main_groups=True,
    make_cog_subgroups=True,
    keep_main_groups_in_subgroup_mode=True,
    localities_to_test=("local", "distant"),

    group_across_sessions=True,
    also_run_per_session=False,
    pooled_output_subdir="pooled_across_sessions",

    save_figures=True,
    figure_dpi=150,
    verbose=True,
)

##### sur une session

In [3]:
# screen des conditions d'une session Hilbert
session_dir = sorted([p for p in Path(hilbert_stats_cfg.input_root).iterdir() if p.is_dir()])[1]
session = session_dir.name
trials_df = pd.read_csv(session_dir / f"{session}_trial_table.csv")

cond_index_main = build_main_condition_index(trials_df, min_trials=hilbert_stats_cfg.min_trials_per_condition)
cond_index_sub = build_cog_subcategory_index(
    trials_df,
    min_trials=hilbert_stats_cfg.min_trials_per_condition,
    keep_main_groups=hilbert_stats_cfg.keep_main_groups_in_subgroup_mode,
)

print({k: len(v) for k, v in cond_index_main.items()})
print({k: len(v) for k, v in cond_index_sub.items()})

{'cog+': 3}
{'cog+': 3, 'cog::illusion_perceptive': 3}


In [25]:
# test d'empilage de Hilbert sur une session
meta = json.load(open(session_dir / f"{session}_metadata.json", "r", encoding="utf-8"))
bp_names = meta["bipolar_names"]

condition_name = list(cond_index_main.keys())[0]
trial_indices = cond_index_main[condition_name]

X_local, obs_local = stack_hilbert_band_condition_locality(
    session=session,
    session_dir=session_dir,
    trials_df=trials_df,
    bp_names=bp_names,
    trial_indices=trial_indices,
    locality="local",
    band_name="low_gamma",
    condition_name=condition_name,
)

print(X_local.shape)   # attendu : (n_observations, n_times)
obs_local.head()

(38, 373)


,session,condition,trial_idx,channel_name,channel_shaft,stim_shaft,label_stim,group_label,cog_labels,locality,band_name
0,P101_DC54_stim2,negatif,31,C1-C2,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],local,low_gamma
1,P101_DC54_stim2,negatif,31,C2-C3,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],local,low_gamma
2,P101_DC54_stim2,negatif,31,C3-C4,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],local,low_gamma
3,P101_DC54_stim2,negatif,31,C4-C5,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],local,low_gamma
4,P101_DC54_stim2,negatif,31,C5-C6,C,C,C_1-C_22.0mA7.0Hz1025µsec,negatif,[],local,low_gamma


In [ ]:
# stats Hilbert sur une seule session
out_session_stats = run_hilbert_session_condition_stats(
    session_dir=session_dir,
    cfg=hilbert_stats_cfg,
)

out_session_stats

NameError: name 'session_dir' is not defined

##### sur toutes les sessions

In [4]:
# test d'empilage de Hilbert pooled across sessions

session_dirs = sorted([p for p in Path(hilbert_stats_cfg.input_root).iterdir() if p.is_dir()])

X_all, obs_all = stack_hilbert_band_condition_locality_across_sessions(
    session_dirs=session_dirs,
    condition_name="cog+",
    locality="distant",
    band_name="theta",
    cfg=hilbert_stats_cfg,
    subgroup_mode=False,
)

print("X_all shape :", X_all.shape)
print("n sessions :", obs_all["session"].nunique())
print("n unique trials :", obs_all[["session", "trial_idx"]].drop_duplicates().shape[0])
print("n unique channels :", obs_all[["session", "channel_name"]].drop_duplicates().shape[0])

obs_all.head()

X_all shape : (243, 384)
n sessions : 1
n unique trials : 3
n unique channels : 85


,session,condition,trial_idx,channel_name,channel_shaft,stim_shaft,label_stim,group_label,cog_labels,locality,band_name
0,P119_FM71_stim4,cog+,16,TPp1-TPp2,TPp,GCp,GCp1-GCp21.8mA50.0Hz1025µsec,cog+,['illusion_perceptive'],distant,theta
1,P119_FM71_stim4,cog+,17,TPp1-TPp2,TPp,CPp,CPp2-CPp32.0mA50.0Hz1025µsec,cog+,['illusion_perceptive'],distant,theta
2,P119_FM71_stim4,cog+,23,TPp1-TPp2,TPp,CPp,CPp2-CPp32.2mA50.0Hz1025µsec,cog+,['illusion_perceptive'],distant,theta
3,P119_FM71_stim4,cog+,16,TPp2-TPp3,TPp,GCp,GCp1-GCp21.8mA50.0Hz1025µsec,cog+,['illusion_perceptive'],distant,theta
4,P119_FM71_stim4,cog+,17,TPp2-TPp3,TPp,CPp,CPp2-CPp32.0mA50.0Hz1025µsec,cog+,['illusion_perceptive'],distant,theta


In [5]:
# stats Hilbert pooled across sessions
out_pooled_stats = run_pooled_hilbert_condition_stats(hilbert_stats_cfg)
out_pooled_stats

/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert_stats/pooled_across_sessions')

In [6]:
# point d'entrée global des stats Hilbert : lance run_pooled_hilbert_condition_stats et/ou run_hilbert_session_condition_stats par session, selon le cfg
summary_hilbert_stats = run_all_hilbert_stats(hilbert_stats_cfg)
summary_hilbert_stats

{'config': {'input_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert',
  'output_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert_stats',
  'bands_to_test': ('theta', 'alpha', 'beta', 'low_gamma', 'high_gamma'),
  'run_wilcoxon_fdr': True,
  'run_cluster_perm': True,
  'alpha_fdr': 0.05,
  'cluster_alpha': 0.05,
  'n_permutations': 2000,
  'cluster_threshold_p': 0.05,
  'tail': 0,
  'seed': 13,
  'min_trials_per_condition': 3,
  'make_main_groups': True,
  'make_cog_subgroups': True,
  'keep_main_groups_in_subgroup_mode': True,
  'localities_to_test': ('local', 'distant'),
  'group_across_sessions': True,
  'also_run_per_session': False,
  'pooled_output_subdir': 'pooled_across_sessions',
  'save_figures': True,
  'figure_dpi': 150,
  'verbose': True},
 'n_errors': 0,
 'errors': []}

In [ ]:
# lecture des résumés produits
summary_file = Path(hilbert_stats_cfg.output_root) / "run_summary_hilbert_stats.json"
with open(summary_file, "r") as f:
    summary = json.load(f)
summary

{'config': {'input_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert',
  'output_root': '/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert_stats',
  'bands_to_test': ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma'],
  'run_wilcoxon_fdr': True,
  'run_cluster_perm': True,
  'alpha_fdr': 0.05,
  'cluster_alpha': 0.05,
  'n_permutations': 2000,
  'cluster_threshold_p': 0.05,
  'tail': 0,
  'seed': 13,
  'min_trials_per_condition': 3,
  'make_main_groups': True,
  'make_cog_subgroups': True,
  'keep_main_groups_in_subgroup_mode': True,
  'localities_to_test': ['local', 'distant'],
  'group_across_sessions': True,
  'also_run_per_session': False,
  'pooled_output_subdir': 'pooled_across_sessions',
  'save_figures': True,
  'figure_dpi': 150,
  'verbose': True},
 'n_errors': 0,
 'errors': []}